# Model Work

In [1]:
import pandas as pd
import numpy as np 
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from imblearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score , davies_bouldin_score
from kmodes.kprototypes import KPrototypes

In [2]:
def run_kprototypes(dataset, categorical_columns, n_clusters=2,
                    random_state=42, gamma=None):
    """Return the fitted model and cluster labels."""

    categorical_indices = [
        dataset.columns.get_loc(col)
        for col in categorical_columns
    ]

    model = KPrototypes(
        n_clusters=n_clusters,
        init="Huang",
        n_init=10,
        random_state=random_state,
        gamma=gamma,
    )

    labels = model.fit_predict(
        dataset.to_numpy(dtype=object),
        categorical=categorical_indices,
    )

    return model, labels


def dunn_index(distances, labels):
    """Calculate Dunn index from a pairwise distance matrix."""

    distances = np.asarray(distances, dtype=float)
    labels = np.asarray(labels)

    if not 2 <= len(np.unique(labels)) < len(labels):
        raise ValueError("Scoring requires 2 to n_samples - 1 clusters.")

    same_cluster = labels[:, None] == labels[None, :]

    smallest_between = distances[~same_cluster].min()
    largest_within = distances[same_cluster].max()

    if largest_within == 0:
        return np.nan  

    return smallest_between / largest_within


def clustering_table(dataset, labels, model, categorical_columns,
                     name="K-prototypes"):
    """Calculate metrics and adding to DataFrame."""

    if not 2 <= len(pd.unique(labels)) < len(dataset):
        raise ValueError("Scoring requires 2 to n_samples - 1 clusters.")

    distances = np.zeros((len(dataset), len(dataset)))

    for col in dataset.columns:
        if col in categorical_columns:
            values = dataset[col].to_numpy()
            distances += model.gamma * (
                values[:, None] != values[None, :]
            )
        else:
            values = dataset[col].to_numpy(dtype=float)
            distances += (
                values[:, None] - values[None, :]
            ) ** 2

    return pd.DataFrame([{
        "Model": name,
        "Clusters": len(pd.unique(labels)),
        "Gamma": model.gamma,
        "Silhouette Score": silhouette_score(
            distances, labels, metric="precomputed"
        ),
        "Dunn Index": dunn_index(distances, labels)
    }])

In [3]:
runs = {}
Scores = pd.DataFrame()
Male_scores = pd.DataFrame()
Female_scores = pd.DataFrame()

In [4]:
encoded_df = pd.read_csv("mental_health_preprocessed.csv")
male_df_encoded = encoded_df.loc[encoded_df["gender_Male"]==True].copy()
female_df_encoded = encoded_df.loc[encoded_df["gender_Female"]==True].copy()

In [5]:
cat_columns = ['academic_year','residential_status', 'campus_discrimination',
       'sports_engagement', 'average_sleep', 'degree_major_Computer Science',
       'degree_major_Data Science', 'degree_major_Information Technology',
       'degree_major_Software Engineering', 'gender_Female', 'gender_Male',
       'activity_Creative Outlets', 'activity_Online Entertainment',
       'activity_Outdoor Activities', 'activity_Religious Activities',
       'activity_Sleep', 'activity_Social Connections',
       'activity_Sports and Fitness', 'study_satisfaction',
       'academic_workload ', 'academic_pressure', 'financial_concerns',
       'social_relationships', 'depression', 'anxiety', 'isolation',
       'future_insecurity']

In [6]:
model_final_E, labels_final_E = run_kprototypes(encoded_df, cat_columns,n_clusters=2, gamma=0.10)

Scores = pd.concat(
    [
       Scores, clustering_table(
            encoded_df, labels_final_E, 
            model_final_E, cat_columns, 
            name = "Total"
        )
    ], ignore_index=True
)

In [7]:
#male final encoded
model_ME, labels_ME = run_kprototypes(male_df_encoded, cat_columns, n_clusters=2, gamma=0.10)

Scores = pd.concat (
    [
        Scores, clustering_table(
    male_df_encoded,
    labels_ME,
    model_ME,
    cat_columns,
    name = "Male"
), 
    ], ignore_index=True
)

In [8]:
#female final encoded
model_FE, labels_FE = run_kprototypes(female_df_encoded, cat_columns, n_clusters=2, gamma=0.10)

Scores = pd.concat (
    [
        Scores, clustering_table(
    female_df_encoded,
    labels_FE,
    model_FE,
    cat_columns,
    name = "Female"
), 
    ], ignore_index=True
)

# Scores

In [9]:
Scores

,Model,Clusters,Gamma,Silhouette Score,Dunn Index
0,Total,2,0.1,0.575431,0.153153
1,Male,2,0.1,0.712791,0.316667
2,Female,2,0.1,0.452450,0.571429


# Comparison Table (code)

In [10]:
# Copies for analysis — original data stays unchanged
overall = encoded_df.copy()
overall["Cluster"] = labels_final_E

male = male_df_encoded.copy()
male["Cluster"] = labels_ME

female = female_df_encoded.copy()
female["Cluster"] = labels_FE


ratings = [
    "study_satisfaction",
    "academic_workload ",
    "academic_pressure",
    "financial_concerns",
    "social_relationships",
    "depression",
    "anxiety",
    "isolation",
    "future_insecurity"
]

groups = {
    "Overall": overall,
    "Male": male,
    "Female": female
}

results = {}

for name, data in groups.items():
    for cluster in sorted(data["Cluster"].unique()):
        group = data.loc[data["Cluster"] == cluster]

        summary = {"Participants": len(group)}

        # Undoing scaling 
        for col in ratings:
            if col == "academic_workload ":
                average = group[col].mean() * 3 + 2
            else:
                average = group[col].mean() * 4 + 1

            label = col.strip().replace("_", " ").capitalize()
            summary[label + " (mean)"] = round(average, 2)

        summary["Sleep: 2–4 hours (%)"] = round(
            group["average_sleep"].eq(0).mean() * 100, 1
        )
        summary["Sleep: 4–6 hours (%)"] = round(
            group["average_sleep"].eq(1).mean() * 100, 1
        )
        summary["Sleep: 7–8 hours (%)"] = round(
            group["average_sleep"].eq(2).mean() * 100, 1
        )

        for col in group.columns:
            if col.startswith("activity_"):
                label = col.replace("activity_", "")
                summary[label + " — coping (%)"] = round(
                    group[col].mean() * 100, 1
                )

        results[f"{name} cluster {cluster}"] = summary

comparison_table = pd.DataFrame(results)

# Comparison Table

In [11]:
comparison_table

,Overall cluster 0,Overall cluster 1,Male cluster 0,Male cluster 1,Female cluster 0,Female cluster 1
Participants,51.00,32.00,11.00,48.00,16.00,8.00
Study satisfaction (mean),4.18,3.44,3.82,3.92,3.69,4.25
Academic workload (mean),3.78,4.03,4.18,3.71,4.06,4.12
Academic pressure (mean),3.63,4.06,3.82,3.65,4.06,4.12
Financial concerns (mean),3.37,3.34,3.36,3.44,3.00,3.62
Social relationships (mean),2.82,2.84,2.73,3.04,2.56,2.25
Depression (mean),3.25,3.03,3.00,3.00,3.38,4.00
Anxiety (mean),3.14,3.25,3.18,3.00,3.19,4.25
Isolation (mean),3.16,3.34,3.55,3.06,3.12,4.00
Future insecurity (mean),2.92,3.03,2.73,2.94,3.25,2.88
